In [ ]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new(target portfolio ).csv',
    encoding='latin-1'
)
print(df.columns)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
pivot_data = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for year in years:
        year_data = company_data[company_data['year'] == year]
        if not year_data.empty:
            row[f'region_{year}'] = year_data['region'].iloc[0]
            row[f'sector_{year}'] = year_data['sector '].iloc[0]
            row[f're100_{year}'] = year_data['re100'].iloc[0]
            row[f'sbti_{year}'] = year_data['sbti'].iloc[0]
            row[f'cn_{year}'] = year_data['cn'].iloc[0]
            row[f'nz_{year}'] = year_data['nz'].iloc[0]
            row[f'cc_{year}'] = year_data['cc'].iloc[0]
    
    pivot_data.append(row)

matrix = pd.DataFrame(pivot_data)
matrix.to_csv('company_matrix.csv', index=False)

print(unique_companies)



Index(['year', 'company ', 'region', 'sector ', 're100', 'sbti', 'cn', 'nz',
       'cc', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11'],
      dtype='object')
0                             3m
1                            abb
2            abbott laboratories
3                         abbvie
4                      accenture
                  ...           
2495                     wistron
2496                   medtronic
2497        china life insurance
2498    china general technology
2499                    heineken
Name: company_canonical, Length: 2500, dtype: object


In [8]:
print(unique_companies)
print(len(unique_companies))


['3m' 'abb' 'abbott laboratories' 'abbvie' 'accenture' 'achmea' 'acs'
 'aegon' 'aeon' 'agricultural bank of china' 'aia group' 'airbus' 'aisin'
 'albertsons' 'alfresa holdings' 'alibaba group holding'
 'alimentation couche-tard' 'allianz' 'allstate' 'alphabet'
 'aluminum corp. of china' 'amazon' 'amer international group'
 'américa móvil' 'american express' 'american international group'
 'amerisourcebergen' 'amgen' 'anglo american' 'anheuser-busch inbev'
 'anhui conch group' 'ansteel group' 'anthem' 'apple' 'arcelormittal'
 'archer daniels midland' 'arrow electronics' 'assicurazioni generali'
 'astrazeneca' 'at&t' 'aviation industry corp. of china' 'aviva' 'axa'
 'bae systems' 'banco bilbao vizcaya argentaria' 'banco bradesco'
 'banco do brasil' 'banco santander' 'bank of america' 'bank of china'
 'bank of communications' 'bank of montreal' 'bank of nova scotia'
 'barclays' 'basf' 'bayer' 'beijing automotive group'
 'beijing jianlong heavy industry group' 'berkshire hathaway' 'best bu

In [9]:
print(company_data)

      year                  company          region  \
2489  2025  GuideWell Mutual Holding  North America   

                              sector   re100  sbti  cn  nz  cc  Unnamed: 9  \
2489  Health care and pharmaceuticals    NaN   NaN NaN NaN NaN         NaN   

      Unnamed: 10  Unnamed: 11        company_normalized  \
2489          NaN          NaN  guidewell mutual holding   

             company_canonical  
2489  guidewell mutual holding  


In [3]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new(target portfolio ).csv',
    encoding='latin-1'
)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']
results = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for target in targets:
        transition = []
        for year in years:
            year_data = company_data[company_data['year'] == year]
            if year_data.empty:
                transition.append('-')
            else:
                val = year_data[target].iloc[0]
                if pd.isna(val):
                    transition.append('0')
                elif val == 1:
                    transition.append('1')
                elif val == -1:
                    transition.append('-1')
                else:
                    transition.append('0')
        row[target] = ''.join(transition)
    
    results.append(row)

output = pd.DataFrame(results)
output.to_csv('company_transitions_map_cc_fixed.csv', index=False)



# SBTI to net zero

In [9]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')
lost_sbti = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    if '10' in sbti:
        lost_sbti.append(row['company'])

print(f"Companies that lost SBTi: {len(lost_sbti)}")
print(lost_sbti)

Companies that lost SBTi: 15
['abbott laboratories', 'bouygues', 'ck hutchison holdings', 'coop group', 'deutsche bank', 'elo group', 'ing group', 'metro', 'mitsui', 'nippon telegraph and telephone', 'sncf group', 'tata motors', 'verizon communications', 'america movil', 'novo nordisk']


In [10]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

lost_sbti = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    if '10' in sbti:
        sbti_loss_pos = sbti.index('10')
        nz_after = row['nz'].replace('-', '')[sbti_loss_pos+1:]
        cn_after = row['cn'].replace('-', '')[sbti_loss_pos+1:]
        
        if '1' in nz_after or '1' in cn_after:
            lost_sbti.append({
                'company': row['company'],
                'sbti': row['sbti'],
                'nz': row['nz'],
                'cn': row['cn']
            })

pd.DataFrame(lost_sbti).to_csv('sbti_lost_then_gained.csv', index=False)

lost_sbti_2023_2025 = []
for _, row in df.iterrows():
    sbti = row['sbti'][2:5].replace('-', '')
    nz = row['nz'][2:5].replace('-', '')
    
    if '10' in sbti:
        lost_pos = sbti.index('10')
        nz_after = nz[lost_pos+1:]
        gained_nz = '1' in nz_after
        lost_sbti_2023_2025.append({'company': row['company'], 'gained_nz': gained_nz})

result = pd.DataFrame(lost_sbti_2023_2025)
print(f"Lost SBTi 2023-2025: {len(result)}")
print(f"Gained NZ: {result['gained_nz'].sum()}")

Lost SBTi 2023-2025: 12
Gained NZ: 3


## fixed 


In [12]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# Total lost SBTi
lost_sbti = 0
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            lost_sbti += 1
            break

print(f"Lost SBTi: {lost_sbti}")

# Lost SBTi and gained NZ or CN
lost_sbti_gained = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    nz = row['nz'].replace('-', '')
    cn = row['cn'].replace('-', '')
    
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            if '1' in nz[i+1:] or '1' in cn[i+1:]:
                lost_sbti_gained.append({
                    'company': row['company'],
                    'sbti': row['sbti'],
                    'nz': row['nz'],
                    'cn': row['cn']
                })
            break

result = pd.DataFrame(lost_sbti_gained)
result.to_csv('sbti_lost_then_gained.csv', index=False)
print(f"Lost SBTi and gained NZ/CN: {len(result)}")

print(lost_sbti)

Lost SBTi: 15
Lost SBTi and gained NZ/CN: 10
15


# carbon neutral to net zero

In [8]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

lost_cn_gained_nz = []
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            if '1' in nz[i+1:]:
                lost_cn_gained_nz.append({'company': row['company'], 'cn': row['cn'], 'nz': row['nz']})
            break

result = pd.DataFrame(lost_cn_gained_nz)
result.to_csv('lost_cn_gained_nz.csv', index=False)
print(result)

lost_cn = 0
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            break

print(f"Lost CN: {lost_cn}")

                             company     cn     nz
0                                abb  11-00  00-10
1                               aeon  10000  01110
2              alibaba group holding  01110  00001
3           alimentation couche-tard  00010  00001
4                            allianz  11000  11110
..                               ...    ...    ...
159  contemporary amperex technology  --010  --001
160                  lufthansa group  --110  --001
161                    tongwei group  --110  --001
162      luxshare precision industry  --110  --001
163                 societe generale  ---10  ---01

[164 rows x 3 columns]
Lost CN: 199


In [ ]:
# Track CN 2021 -> NZ 2025
cn_to_nz = matrix[
    (matrix['cn_2021'] == 1) & 
    (matrix['nz_2025'] == 1)
]['company'].tolist()

# Track SBTi changes every two years
sbti_transitions = {}
for i in range(len(years) - 1):
    year1, year2 = years[i], years[i+1]
    if year2 - year1 <= 2:
        gained = matrix[
            (matrix[f'sbti_{year1}'] != 1) & 
            (matrix[f'sbti_{year2}'] == 1)
        ]['company'].tolist()
        lost = matrix[
            (matrix[f'sbti_{year1}'] == 1) & 
            (matrix[f'sbti_{year2}'] != 1)
        ]['company'].tolist()
        sbti_transitions[f'{year1}_to_{year2}'] = {'gained': gained, 'lost': lost}

print(f"Companies with CN in 2021 and NZ in 2025: {len(cn_to_nz)}")
print(f"\nSBTi transitions: {sbti_transitions}")

matrix.to_csv('company_matrix.csv', index=False)

that brings the 

In [13]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

print(df.iloc[489:639])

                           company  re100   sbti     cn     nz     cc
489              sinochem holdings  -000-  -000-  -000-  -000-  -000-
490            mercedes-benz group  -0000  -0011  -1110  -0001  -1010
491                elevance health  -1111  -0001  -0000  -0110  -0101
492                 meta platforms  -1110  -0011  -0000  -1110  -0011
493  life insurance corp. of india  -0000  -0000  -0000  -0000  -0000
..                             ...    ...    ...    ...    ...    ...
632   international airlines group  ----0  ----0  ----0  ----0  ----1
633   pnc financial services group  ----0  ----0  ----0  ----0  ----1
634      perusahaan listrik negara  ----0  ----0  ----0  ----0  ----1
635              st. james's place  ----0  ----0  ----0  ----0  ----1
636       guidewell mutual holding  ----0  ----0  ----0  ----0  ----0

[148 rows x 6 columns]


## dealing with normalization and false exits/entries:





#### Category 1: Company Rebrands (7 cases)

1. Royal Dutch Shell → Shell

Old name: royal dutch shell (2019-2021)
New name: shell (2022-2025)
Change year: 2021
Impact on data: Shell data starts in 2022 with NZ commitment

2. Facebook → Meta Platforms

Old name: facebook (2019-2021)
New name: meta platforms (2022-2025)
Change year: October 2021
Impact on data: Meta shows sustainability commitments starting 2022

3. Daimler → Mercedes-Benz Group

Old name: daimler (2019-2021)
New name: mercedes-benz group (2022-2025)
Change year: 2022
Impact on data: Mercedes-Benz shows comprehensive commitments from 2023+

4. GlaxoSmithKline → GSK

Old name: glaxosmithkline (2019-2021)
New name: gsk (2022-2025)
Change year: July 2022
Impact on data: GSK continues strong commitments from earlier period

5. Raytheon Technologies → RTX

Old name: raytheon technologies (2019-2022)
New name: rtx (2023-2025)
Change year: July 2023
Impact on data: RTX shows NZ commitment starting 2024

6. ViacomCBS → Paramount Global

Old name: viacomcbs (2019-2021)
New name: paramount global (2022-2025)
Change year: February 2022
Impact on data: Both show no commitments (all zeros/dashes)

7. Anthem → Elevance Health

Old name: anthem (2019-2021)
New name: elevance health (2022-2025)
Change year: June 2022
Impact on data: Elevance shows multiple commitments starting 2022


Category 2: Holding Company Structures (4 cases)
8. Panasonic / Panasonic Holdings

Pattern: Original "Panasonic" (2019-2021) → "Panasonic Holdings" (2022-2025)
Reason: Holding company restructuring in 2022
Impact: Panasonic Holdings shows continued commitments

9. POSCO / POSCO Holdings

Pattern: "POSCO" (2019-2022) → "POSCO Holdings" (2023-2025)
Reason: Holding company restructuring
Impact: Some data overlap in transition years

10. SK / SK Group

Pattern: Both names appear across different years
Reason: Holding company structure
Impact: Need to merge these records

11. Sinochem / Sinochem Holdings

Pattern: "Sinochem" (2019-2021) → "Sinochem Holdings" (2022-2024)
Reason: Holding company restructuring
Impact: All zeros across both entries


Category 3: Accent Variations (4 cases)
12. Nestlé / Nestle

Pattern: "Nestlé" (2019-2023) and "Nestle" (2024-2025)
Reason: Database encoding differences
Impact: SAME COMPANY - data should be continuous

13. América Móvil / America Movil

Pattern: Both spellings used
Reason: Accent handling in data entry
Impact: SAME COMPANY - merge needed

14. Raízen / Raizen

Pattern: Both spellings used
Reason: Accent handling
Impact: SAME COMPANY - no commitments in either

15. Société Générale / Societe Generale

Pattern: Both spellings used
Reason: Accent handling
Impact: SAME COMPANY - some CN commitments


Category 4: Abbreviations (2 cases)
16. Nippon Telegraph and Telephone / NTT

Full name: nippon telegraph and telephone (2019-2023)
Abbreviation: ntt (2025)
Impact: SAME COMPANY - continuous operations

17. International Business Machines / IBM

Full name: international business machines (2019-2022)
Abbreviation: ibm (2023-2025)
Impact: SAME COMPANY - IBM shows NZ commitments from 2024


Category 5: Name Simplifications (2 cases)
18. Deutsche Post DHL Group / DHL Group

Old name: deutsche post dhl group (2019-2022)
New name: dhl group (2023-2025)
Impact: Name simplification, continuous operations

19. PKN Orlen Group / Orlen

Old name: pkn orlen group (2019-2022)
New name: orlen (2023-2025)
Impact: Name simplification


Category 6: Mergers & Acquisitions (1 case)
20. Synnex → TD Synnex

Old name: synnex (2019-2021)
New name: td synnex (2022-2025)
Merger: Tech Data Corporation merged with Synnex in 2021
Impact: TD Synnex shows commitments starting 2023


Category 7: Company Splits (1 case)
21. General Electric Split

Original: general electric (2019-2023)
Split into:

general electric (ge aerospace) (2024-2025)
ge vernova (2025)


Split year: 2024
Impact: GE split into three companies (Healthcare spun off as GE HealthCare in 2023, then GE split into Aerospace and Vernova in 2024)


Category 8: Other Variations (6 cases)
22. Mitsubishi / Mitsubishi Corp

Pattern: Both names used
Impact: Likely different entities (Mitsubishi Corporation vs Mitsubishi Group)
Note: May actually be different companies

23. Nippon Steel / Nippon Steel Corporation

Pattern: "Nippon Steel Corporation" (2019-2023), "Nippon Steel" (2025)
Impact: SAME COMPANY - name variation

24. Bunge / Bunge Global

Pattern: "Bunge" (2019-2023), "Bunge Global" (2025)
Impact: Name change to Bunge Global in 2024

25. China COSCO Shipping / COSCO Shipping

Pattern: "China COSCO Shipping" (2019-2021), "COSCO Shipping" (2022-2025)
Impact: Name simplification

26. Olam International / Olam Group

Pattern: "Olam International" (2019-2021), "Olam Group" (2022-2025)
Impact: Restructuring in 2022

27. AmerisourceBergen / Cencora

Old name: amerisourcebergen (2019-2022)
New name: cencora (2023-2025)
Change year: August 2023
Impact: Company rebrand


#### name differences 

High Priority (MUST MERGE)
These are definitely the same company and should be treated as single entities:

Royal Dutch Shell / Shell
Facebook / Meta Platforms
GlaxoSmithKline / GSK
Anthem / Elevance Health
Raytheon Technologies / RTX
Daimler / Mercedes-Benz Group
AmerisourceBergen / Cencora
Nestlé / Nestle (accent variation)
América Móvil / America Movil (accent variation)
Société Générale / Societe Generale (accent variation)
Raízen / Raizen (accent variation)
Nippon Telegraph and Telephone / NTT
International Business Machines / IBM
Deutsche Post DHL Group / DHL Group
PKN Orlen Group / Orlen
Synnex / TD Synnex
Bunge / Bunge Global
China COSCO Shipping / COSCO Shipping
Olam International / Olam Group
Nippon Steel / Nippon Steel Corporation

#### holding company decisions 
Medium Priority (HOLDING COMPANY DECISIONS)
These require a decision on whether to treat as single entity or separate:

Panasonic / Panasonic Holdings
POSCO / POSCO Holdings
SK / SK Group
Sinochem / Sinochem Holdings

Recommendation: Treat as single company for sustainability analysis, as holding companies typically consolidate environmental commitments.

#### company split 
Special Case (COMPANY SPLIT)

General Electric → GE Aerospace + GE Vernova

Recommendation:

Keep pre-2024 data under "General Electric"
Track 2024+ separately for split entities
Note that sustainability commitments may have transferred to new entities

Uncertain (NEED VERIFICATION)

Mitsubishi / Mitsubishi Corp

Recommendation: Verify if these are truly different entities (Mitsubishi Corporation vs Mitsubishi Group companies)


( yes truly diff after verification)

In [15]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# Define rebrands: (old_name, new_name, final_name)
rebrands = [
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
]

# Print comparison
for old, new, final in rebrands:
    old_row = df[df['company'].str.lower() == old].iloc[0]
    new_row = df[df['company'].str.lower() == new].iloc[0]
    
    print(f"\n{old.upper()} → {new.upper()} = {final.upper()}")
    print(f"       RE100  SBTi   CN     NZ     CC")
    print(f"Old:   {old_row['re100']}  {old_row['sbti']}  {old_row['cn']}  {old_row['nz']}  {old_row['cc']}")
    print(f"New:   {new_row['re100']}  {new_row['sbti']}  {new_row['cn']}  {new_row['nz']}  {new_row['cc']}")

# Merge function
def merge_strings(s1, s2):
    return ''.join(c1 if c1 != '-' else c2 for c1, c2 in zip(s1 + '-'*len(s2), s2 + '-'*len(s1)))[:max(len(s1), len(s2))]




ROYAL DUTCH SHELL → SHELL = SHELL
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  0----  1----  0----
New:   -0000  -0000  -0000  -1110  -1011

FACEBOOK → META PLATFORMS = META PLATFORMS
       RE100  SBTi   CN     NZ     CC
Old:   1----  0----  0----  1----  0----
New:   -1110  -0011  -0000  -1110  -0011

DAIMLER → MERCEDES-BENZ GROUP = MERCEDES-BENZ GROUP
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  1----  0----  0----
New:   -0000  -0011  -1110  -0001  -1010

GLAXOSMITHKLINE → GSK = GSK
       RE100  SBTi   CN     NZ     CC
Old:   1----  1----  0----  1----  0----
New:   -0111  -1111  -0000  -1110  -0011

RAYTHEON TECHNOLOGIES → RTX = RTX
       RE100  SBTi   CN     NZ     CC
Old:   00---  00---  00---  00---  00---
New:   --000  --000  --000  --110  --010

VIACOMCBS → PARAMOUNT GLOBAL = PARAMOUNT GLOBAL
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  0----  0----  0----
New:   -00--  -00--  -00--  -00--  -00--

ANTHEM → ELEVANCE HEALTH = ELEVANC

In [17]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

rebrands = [
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
]

def merge_strings(s1, s2):
    max_len = max(len(s1), len(s2))
    result = []
    for i in range(max_len):
        c1 = s1[i] if i < len(s1) else '-'
        c2 = s2[i] if i < len(s2) else '-'
        result.append(c1 if c1 != '-' else c2)
    return ''.join(result)

merged_rows = []
for old, new, final in rebrands:
    old_row = df[df['company'].str.lower() == old].iloc[0]
    new_row = df[df['company'].str.lower() == new].iloc[0]
    
    merged_rows.append({
        'company': final,
        're100': merge_strings(old_row['re100'], new_row['re100']),
        'sbti': merge_strings(old_row['sbti'], new_row['sbti']),
        'cn': merge_strings(old_row['cn'], new_row['cn']),
        'nz': merge_strings(old_row['nz'], new_row['nz']),
        'cc': merge_strings(old_row['cc'], new_row['cc']),
    })
    
    print(f"{final}: RE100={merged_rows[-1]['re100']} NZ={merged_rows[-1]['nz']}")

old_new_names = [old for old, new, _ in rebrands] + [new for _, new, _ in rebrands]
df_final = df[~df['company'].str.lower().isin([n.lower() for n in old_new_names])]
df_final = pd.concat([df_final, pd.DataFrame(merged_rows)], ignore_index=True)
df_final = df_final.sort_values('company').reset_index(drop=True)

df_final.to_csv('company_data_rebrands_merged.csv', index=False)
print(f"\nOriginal: {len(df)} → Merged: {len(df_final)} (removed {len(df)-len(df_final)} duplicates)")

Shell: RE100=00000 NZ=11110
Meta Platforms: RE100=11110 NZ=11110
Mercedes-Benz Group: RE100=00000 NZ=00001
GSK: RE100=10111 NZ=11110
RTX: RE100=00000 NZ=00110
Paramount Global: RE100=000-- NZ=000--
Elevance Health: RE100=11111 NZ=00110
Cencora: RE100=00000 NZ=00000
TD Synnex: RE100=00000 NZ=01110
Bunge Global: RE100=00000 NZ=00000
COSCO Shipping: RE100=00000 NZ=00110
Olam Group: RE100=00000 NZ=00010

Original: 637 → Merged: 625 (removed 12 duplicates)


In [18]:

# All merges: (old_name, new_name, final_name)
merges = [
    # Rebrands
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
    # Holding companies
    ('panasonic', 'panasonic holdings', 'Panasonic Holdings'),
    ('posco', 'posco holdings', 'POSCO Holdings'),
    ('sk', 'sk group', 'SK Group'),
    ('sinochem', 'sinochem holdings', 'Sinochem Holdings'),
    # Accent variations
    ('nestlé', 'nestle', 'Nestlé'),
    ('américa móvil', 'america movil', 'América Móvil'),
    ('raízen', 'raizen', 'Raízen'),
    ('société générale', 'societe generale', 'Société Générale'),
    # Abbreviations
    ('nippon telegraph and telephone', 'ntt', 'NTT'),
    ('international business machines', 'ibm', 'IBM'),
    # Name simplifications
    ('deutsche post dhl group', 'dhl group', 'DHL Group'),
    ('pkn orlen group', 'orlen', 'Orlen'),
    # Other
    ('nippon steel corporation', 'nippon steel', 'Nippon Steel'),
]

def merge_strings(s1, s2):
    max_len = max(len(s1), len(s2))
    result = []
    for i in range(max_len):
        c1 = s1[i] if i < len(s1) else '-'
        c2 = s2[i] if i < len(s2) else '-'
        result.append(c1 if c1 != '-' else c2)
    return ''.join(result)

merged_rows = []
for old, new, final in merges:
    old_row = df[df['company'].str.lower() == old]
    new_row = df[df['company'].str.lower() == new]
    
    if len(old_row) > 0 and len(new_row) > 0:
        old_row = old_row.iloc[0]
        new_row = new_row.iloc[0]
        
        merged_rows.append({
            'company': final,
            're100': merge_strings(old_row['re100'], new_row['re100']),
            'sbti': merge_strings(old_row['sbti'], new_row['sbti']),
            'cn': merge_strings(old_row['cn'], new_row['cn']),
            'nz': merge_strings(old_row['nz'], new_row['nz']),
            'cc': merge_strings(old_row['cc'], new_row['cc']),
        })

old_new_names = [old for old, new, _ in merges] + [new for _, new, _ in merges]
df_final = df[~df['company'].str.lower().isin([n.lower() for n in old_new_names])]
df_final = pd.concat([df_final, pd.DataFrame(merged_rows)], ignore_index=True)
df_final = df_final.sort_values('company').reset_index(drop=True)

df_final.to_csv('company_data_all_merged.csv', index=False)
print(f"{len(df)} → {len(df_final)} ({len(df)-len(df_final)} merged)")

637 → 612 (25 merged)


In [25]:
import pandas as pd

df = pd.read_csv('company_data_all_merged.csv')

# Total lost SBTi
lost_sbti = 0
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            lost_sbti += 1
            break

print(f"Lost SBTi: {lost_sbti}")

# Lost SBTi and gained NZ or CN
lost_sbti_gained = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    nz = row['nz'].replace('-', '')
    cn = row['cn'].replace('-', '')
    
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            if '1' in nz[i+1:] or '1' in cn[i+1:]:
                lost_sbti_gained.append({
                    'company': row['company'],
                    'sbti': row['sbti'],
                    'nz': row['nz'],
                    'cn': row['cn']
                })
            break

result = pd.DataFrame(lost_sbti_gained)
result.to_csv('sbti_lost_then_gained.csv', index=False)
print(f"Lost SBTi and gained NZ/CN: {len(result)}")

print(lost_sbti)

rate = len(lost_sbti_gained) / lost_sbti if lost_sbti != 0 else 0
print(f"transition rate: {rate:.4f}")


Lost SBTi: 15
Lost SBTi and gained NZ/CN: 11
15
transition rate: 0.7333


In [27]:
import pandas as pd

df = pd.read_csv('company_data_all_merged.csv')

def clean(s):
    return s.replace('-', '')

# Year 1→3
sbti_lost_y13 = 0
sbti_lost_gained_y13 = 0
for _, row in df.iterrows():
    sbti = clean(row['sbti'])[:3]
    nz = clean(row['nz'])[:3]
    cn = clean(row['cn'])[:3]
    if '1' in sbti:
        loss_idx = sbti.index('1')
        if '0' in sbti[loss_idx+1:]:
            sbti_lost_y13 += 1
            if '1' in nz[loss_idx+1:] or '1' in cn[loss_idx+1:]:
                sbti_lost_gained_y13 += 1

# Year 3→5
sbti_lost_y35 = 0
sbti_lost_gained_y35 = 0
for _, row in df.iterrows():
    sbti = clean(row['sbti'])[2:5]
    nz = clean(row['nz'])[2:5]
    cn = clean(row['cn'])[2:5]
    if '1' in sbti:
        loss_idx = sbti.index('1')
        if '0' in sbti[loss_idx+1:]:
            sbti_lost_y35 += 1
            if '1' in nz[loss_idx+1:] or '1' in cn[loss_idx+1:]:
                sbti_lost_gained_y35 += 1

print(f"Year 1→3: {sbti_lost_y13} lost SBTi, {sbti_lost_gained_y13} gained NZ/CN ({sbti_lost_gained_y13/sbti_lost_y13:.1%})")
print(f"Year 3→5: {sbti_lost_y35} lost SBTi, {sbti_lost_gained_y35} gained NZ/CN ({sbti_lost_gained_y35/sbti_lost_y35:.1%})")

Year 1→3: 5 lost SBTi, 4 gained NZ/CN (80.0%)
Year 3→5: 11 lost SBTi, 7 gained NZ/CN (63.6%)


CN NZ

In [28]:

import pandas as pd

df = pd.read_csv('company_data_all_merged.csv')

lost_cn_gained_nz = []
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            if '1' in nz[i+1:]:
                lost_cn_gained_nz.append({'company': row['company'], 'cn': row['cn'], 'nz': row['nz']})
            break

result = pd.DataFrame(lost_cn_gained_nz)
result.to_csv('lost_cn_gained_nz.csv', index=False)
print(result)

lost_cn = 0
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            break

print(f"Lost CN: {lost_cn}")

                          company     cn     nz
0                   América Móvil  10000  01100
1             Mercedes-Benz Group  11110  00001
2                    Nippon Steel  11110  00001
3                           Orlen  -0100  -1010
4                  POSCO Holdings  11010  00100
..                            ...    ...    ...
161                        xiaomi  00010  00001
162            zf friedrichshafen  11010  00101
163  zhejiang geely holding group  01110  00001
164            zijin mining group  01110  00001
165        zurich insurance group  11000  11110

[166 rows x 3 columns]
Lost CN: 201


In [32]:
import pandas as pd

df = pd.read_csv('company_data_all_merged.csv')

lost_cn = 0
lost_cn_gained_nz = 0

for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            if '1' in nz[i+1:]:
                lost_cn_gained_nz += 1
            break

print(f"Lost CN: {lost_cn}, gained NZ: {lost_cn_gained_nz} ({lost_cn_gained_nz/lost_cn:.1%})")


Lost CN: 201, gained NZ: 166 (82.6%)
